# SGJobData — Step-by-Step Data Cleaning

Loads `data/raw/SGJobData.csv` and cleans it one documented step at a time.
Every step prints a **BEFORE / AFTER** comparison so the effect of the change is visible,
and records its numbers into the `M` metrics dict.

The final cell writes `docs/data-cleaning-report-generated.md` from those live numbers, with
sections for **type conversions**, **ghost rows**, **data fixing**, **data filling**,
**duplicate rows** and **feature enrichment**.

Nothing here is hard-coded from a previous run: re-running on a refreshed extract produces a
report that matches the new data.

**Pipeline order** — junk rows first (so column statistics are computed on real postings),
then structure, then values, then dtypes:

```
load → ghost rows → synthetic rows → prune dead columns → dates → categories JSON
     → salary fixes → text normalisation → filling decisions → duplicates
     → categorical + downcast → validate + save → feature enrichment → report
```

Outputs `jobs_clean.parquet` (source column names), `job_category.parquet` (the many-to-many
bridge) and `jobs_enriched.parquet` (renamed and derived to match `src/pipeline/` and `JOBS_SCHEMA`).

## 0 · Setup

`SALARY_FLOOR` and `SALARY_CEILING` are the two judgement calls in the whole pipeline.
They are declared here, in one place, so the report can state exactly what was assumed.

In [1]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd

RAW_CSV     = Path('../data/raw/SGJobData.csv')
OUT_DIR     = Path('../data/processed')
REPORT_PATH = Path('../docs/data-cleaning-report-generated.md')
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- tunable cleaning parameters (surfaced in the report) ---------------------
SALARY_FLOOR         = 500      # monthly SGD; at or below this a salary is a placeholder, not a wage
SALARY_CEILING       = 100_000  # monthly SGD; above this it is a data-entry error
INTERN_STIPEND_FLOOR = 300      # monthly SGD; below SALARY_FLOOR but a plausible internship stipend
SYNTHETIC_ID_RE      = r'^RANDOM_JOB_'
EXPERIENCE_MAX       = 100       # years; above this the value is not physically possible

# --- JOBS_SCHEMA columns that are dead in THIS extract, and why -------------------------------
# A cleaning decision, so it is declared here with the others: whether a column is worth
# materialising is a property of the data, not of the enrichment code, which simply produces a
# column list that does not include them. Checked against both outputs in Step 12 and Step 13.
DEAD_COLUMNS = {
    'location':        'the extract is Singapore-only - zero variance',
    'salary_currency': "restates salary_type, itself constant 'Monthly' here and dropped in Step 4",
    'description':     'the source CSV has no description field at all',
    'requirements':    'the source CSV has no requirements field at all',
    'created_at':      'records when the pipeline ran, not a property of the posting; '
                       'DatabaseManager.insert_jobs stamps it at insert time',
}

pd.set_option('display.width', 200, 'display.max_columns', 60, 'display.max_rows', 80)

M = {'steps': [], 'params': {'SALARY_FLOOR': SALARY_FLOOR, 'SALARY_CEILING': SALARY_CEILING,
                             'INTERN_STIPEND_FLOOR': INTERN_STIPEND_FLOOR,
                             'SYNTHETIC_ID_RE': SYNTHETIC_ID_RE, 'EXPERIENCE_MAX': EXPERIENCE_MAX},
     'dead_schema_cols': dict(DEAD_COLUMNS)}
print('pandas', pd.__version__, '| numpy', np.__version__)
print('dead columns (never written to either output):')
for _c, _why in DEAD_COLUMNS.items():
    print(f'   {_c:<16} {_why}')

pandas 3.0.5 | numpy 2.4.6
dead columns (never written to either output):
   location         the extract is Singapore-only - zero variance
   salary_currency  restates salary_type, itself constant 'Monthly' here and dropped in Step 4
   description      the source CSV has no description field at all
   requirements     the source CSV has no requirements field at all
   created_at       records when the pipeline ran, not a property of the posting; DatabaseManager.insert_jobs stamps it at insert time


### Comparison helpers

`snap()` takes a cheap fingerprint of a frame (rows, columns, memory, NaN cells) instead of a
full `.copy()`, so a 400 MB frame can be compared at every step without doubling memory.
`compare()` prints the before/after block and files the delta into `M['steps']` for the report.

In [2]:
def snap(d):
    'Cheap fingerprint of a DataFrame - avoids copying 400 MB at every step.'
    return {'rows': len(d), 'cols': d.shape[1],
            'mem': round(d.memory_usage(deep=True).sum() / 1e6, 1),
            'na': int(d.isna().sum().sum()), 'columns': list(d.columns)}


def compare(before, after, title, note=''):
    'Print a BEFORE/AFTER block for one cleaning step and record it in M.'
    a = after if isinstance(after, dict) else snap(after)
    print(f'=== {title} ===')
    if note:
        print(f'    {note}')
    hdr = f'{"":<7} {"rows":>11} {"cols":>6} {"mem MB":>9} {"NaN cells":>12}'
    print(hdr)
    print(f'{"BEFORE":<7} {before["rows"]:>11,} {before["cols"]:>6} {before["mem"]:>9,.1f} {before["na"]:>12,}')
    print(f'{"AFTER":<7} {a["rows"]:>11,} {a["cols"]:>6} {a["mem"]:>9,.1f} {a["na"]:>12,}')
    print(f'{"DELTA":<7} {a["rows"]-before["rows"]:>+11,} {a["cols"]-before["cols"]:>+6} '
          f'{a["mem"]-before["mem"]:>+9,.1f} {a["na"]-before["na"]:>+12,}')
    removed = [c for c in before['columns'] if c not in a['columns']]
    added   = [c for c in a['columns'] if c not in before['columns']]
    if removed: print('    columns removed:', removed)
    if added:   print('    columns added  :', added)
    print()
    M['steps'].append({'step': title, 'note': note,
                       'rows_before': before['rows'], 'rows_after': a['rows'],
                       'cols_before': before['cols'], 'cols_after': a['cols'],
                       'mem_before': before['mem'], 'mem_after': a['mem'],
                       'removed': removed, 'added': added})
    return a


def to_md(d, index=True):
    'DataFrame -> markdown table (avoids a tabulate dependency).'
    d = d.reset_index() if index else d
    cols = [str(c) for c in d.columns]

    def fmt(v):
        if isinstance(v, (bool, np.bool_)):        return str(bool(v))
        if isinstance(v, float) and not np.isnan(v):
            return f'{v:,.2f}'.rstrip('0').rstrip('.') if abs(v) < 1e15 else f'{v:,.0f}'
        if isinstance(v, (int, np.integer)):       return f'{v:,}'
        return str(v)

    body = '\n'.join('| ' + ' | '.join(fmt(v) for v in rec) + ' |' for rec in d.itertuples(index=False))
    return ('| ' + ' | '.join(cols) + ' |\n'
            '| ' + ' | '.join('---' for _ in cols) + ' |\n' + body)


def profile(d):
    'Per-column dtype / missingness / zero-inflation profile.'
    num = d.select_dtypes(include='number').columns
    return pd.DataFrame({
        'dtype':    d.dtypes.astype(str),
        'na_count': d.isna().sum(),
        'na_pct':   (d.isna().mean() * 100).round(2),
        'n_unique': d.nunique(dropna=True),
        'zeros':    [int((d[c] == 0).sum()) if c in num else 0 for c in d.columns],
    })

## 1 · Load the raw CSV

`read_csv` with no `dtype=` map, deliberately — the point of this notebook is to show what the
untyped load costs and to fix it explicitly rather than hide it in a parser argument.

In [3]:
df = pd.read_csv(RAW_CSV)
s0 = snap(df)
M['raw'] = s0

print(f'loaded {s0["rows"]:,} rows x {s0["cols"]} cols, {s0["mem"]:,.1f} MB deep')

# baseline for the float32 question revisited in Step 11 - measured on untouched raw values
M['float32_raw'] = {
    'max_error': float((df['average_salary'].astype('float32').astype('float64')
                        - df['average_salary']).abs().max()),
    'max_value': float(df['average_salary'].max()),
}
print(f'float32 round-trip error on raw average_salary: {M["float32_raw"]["max_error"]} '
      f'(max value ${M["float32_raw"]["max_value"]:,.0f})')

prof_raw = profile(df)
M['md_profile_raw'] = to_md(prof_raw.sort_values('na_count', ascending=False))
prof_raw.sort_values('na_count', ascending=False)

loaded 1,048,585 rows x 22 cols, 401.7 MB deep
float32 round-trip error on raw average_salary: 0.5 (max value $12,666,400)


,dtype,na_count,na_pct,n_unique,zeros
occupationId,float64,1048585,100.00,0,0
categories,str,3988,0.38,21125,0
metadata_expiryDate,str,3988,0.38,453,0
title,str,3988,0.38,377084,0
metadata_jobPostId,str,3988,0.38,1044597,0
metadata_newPostingDate,str,3988,0.38,431,0
metadata_originalPostingDate,str,3988,0.38,603,0
status_jobStatus,str,3988,0.38,3,0
salary_type,str,3988,0.38,1,0
employmentTypes,str,3988,0.38,8,0


## 2 · Ghost rows

A *ghost row* is a structurally empty record: every text field `NaN` **and** every numeric field
exactly `0`. The evidence for treating them as junk rather than as partially-missing postings is
that the per-row NaN count is **bimodal** — a row is either fully populated or fully blank, never
in between. If these were real postings with patchy collection, we would see intermediate counts.

They are removed first so that every column statistic computed later describes real postings.

> Note the trap this avoids: `occupationId` is 100% NaN, so `df.dropna()` on the raw frame
> returns an **empty** DataFrame. An explicit mask states the intent and cannot misfire.

In [4]:
num_cols  = list(df.select_dtypes(include='number').columns)
bool_cols = list(df.select_dtypes(include='bool').columns)
text_cols = [c for c in df.columns if c not in num_cols + bool_cols]

na_per_row = df[text_cols].isna().sum(axis=1)
print('NaN-count-per-row distribution across the', len(text_cols), 'text columns:')
print(na_per_row.value_counts().sort_index().to_string(), '\n')

ghost = (na_per_row == len(text_cols)) & (df[num_cols].fillna(0) == 0).all(axis=1)

M['ghost'] = {
    'n': int(ghost.sum()),
    'pct': round(ghost.mean() * 100, 3),
    'text_cols': len(text_cols),
    'bimodal': sorted(int(v) for v in na_per_row.unique()),
    'numeric_abs_sum': float(df.loc[ghost, num_cols].abs().sum().sum()),
    'idx_min': int(df.index[ghost].min()), 'idx_max': int(df.index[ghost].max()),
    'partial': int(((na_per_row > 0) & (na_per_row < len(text_cols))).sum()),
    'vacancies_zero_elsewhere': int(((df['numberOfVacancies'] == 0) & ~ghost).sum()),
}
print('ghost rows                      :', f'{M["ghost"]["n"]:,}', f'({M["ghost"]["pct"]}%)')
print('rows with a PARTIAL NaN pattern :', M['ghost']['partial'], '<- 0 proves the pattern is all-or-nothing')
print('sum |numeric| over ghost rows   :', M['ghost']['numeric_abs_sum'], '<- every numeric field is exactly 0')
print('rows with 0 vacancies elsewhere :', M['ghost']['vacancies_zero_elsewhere'])
print('index span                      :', M['ghost']['idx_min'], '-', M['ghost']['idx_max'])

before = snap(df)
df = df.loc[~ghost].copy()
compare(before, df, 'Step 2 - remove ghost rows',
        f'{M["ghost"]["n"]:,} structurally empty records dropped')

NaN-count-per-row distribution across the 11 text columns:
0     1044597
11       3988 

ghost rows                      : 3,988 (0.38%)
rows with a PARTIAL NaN pattern : 0 <- 0 proves the pattern is all-or-nothing
sum |numeric| over ghost rows   : 0.0 <- every numeric field is exactly 0
rows with 0 vacancies elsewhere : 0
index span                      : 197478 - 606701


=== Step 2 - remove ghost rows ===
    3,988 structurally empty records dropped
               rows   cols    mem MB    NaN cells
BEFORE    1,048,585     22     401.7    1,092,453
AFTER     1,044,597     22     409.4    1,044,597
DELTA        -3,988     +0      +7.7      -47,856



{'rows': 1044597,
 'cols': 22,
 'mem': np.float64(409.4),
 'na': 1044597,
 'columns': ['categories',
  'employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'occupationId',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'salary_type',
  'status_id',
  'status_jobStatus',
  'title',
  'average_salary']}

## 3 · Synthetic test rows

Rows whose `metadata_jobPostId` matches `RANDOM_JOB_*` are generated test data, not MCF postings:
the IDs embed a generation timestamp and the salaries are impossible. They are removed by **ID
pattern**, not by index, so the filter survives a reload or a re-sorted extract.

The `ATS-` prefixed rows are kept — they are a legitimate second source (real companies, sane
salaries) and get recorded in a new `source` column later.

In [5]:
synthetic = df['metadata_jobPostId'].str.match(SYNTHETIC_ID_RE, na=False)
prefix = df['metadata_jobPostId'].str.extract(r'^([A-Za-z_]+)')[0].value_counts()

print('ID prefixes present:')
print(prefix.to_string(), '\n')
print('synthetic rows to remove:', int(synthetic.sum()))
if synthetic.any():
    print(df.loc[synthetic, ['metadata_jobPostId', 'title', 'salary_minimum', 'salary_maximum']]
            .to_string(index=False))

M['synthetic'] = {'n': int(synthetic.sum()), 'prefixes': prefix.to_dict(),
                  'max_salary': int(df.loc[synthetic, 'salary_maximum'].max()) if synthetic.any() else 0}

before = snap(df)
df = df.loc[~synthetic].copy()
compare(before, df, 'Step 3 - remove synthetic test rows',
        f'{M["synthetic"]["n"]} RANDOM_JOB_* rows dropped')

ID prefixes present:
0
MCF            1044463
ATS                124
RANDOM_JOB_         10 

synthetic rows to remove: 10
               metadata_jobPostId                                                                        title  salary_minimum  salary_maximum
RANDOM_JOB_20251115011346015685_0                                                  Senior Manager - Operations          107908         6142101
RANDOM_JOB_20251115011346673349_1                                                           Resident Physician          108872        10734314
RANDOM_JOB_20251115011347120191_2 Senior Logistics Executive (1 yr contract) - up to $5k/West/SCIENCE MNC #HAO          276583         2720804
RANDOM_JOB_20251115011347466118_3                                                             Language Teacher          324072        20862169
RANDOM_JOB_20251115011347817248_4                                         Sales Associate (Home Audio, Retail)          262482        15531134
RANDOM_JOB_20251115

=== Step 3 - remove synthetic test rows ===
    10 RANDOM_JOB_* rows dropped
               rows   cols    mem MB    NaN cells
BEFORE    1,044,597     22     409.4    1,044,597
AFTER     1,044,587     22     409.4    1,044,587
DELTA           -10     +0      +0.0          -10



{'rows': 1044587,
 'cols': 22,
 'mem': np.float64(409.4),
 'na': 1044587,
 'columns': ['categories',
  'employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'occupationId',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'salary_type',
  'status_id',
  'status_jobStatus',
  'title',
  'average_salary']}

## 4 · Prune dead columns

Detected, not hard-coded: a column is dead if it is entirely empty — every value `NaN` *or* a
blank string, since `''` is not `NaN` and would otherwise pass an `isna()` check — or if it holds a
single distinct value across a million real rows. A constant column cannot correlate with anything,
cannot be filtered on, and costs memory on every read.

The same rule is applied at the other end of the pipeline: `DEAD_COLUMNS` (Step 13) names the
`JOBS_SCHEMA` fields that would be constant or blank if they were materialised, so they are never
created in the first place.

`salary_type` being constant is information — it means *all salaries are monthly SGD* — so it is
recorded in the report rather than repeated a million times in the frame.

In [6]:
def all_blank(s):
    'True if every value is NaN or an empty / whitespace-only string - a column with no content.'
    # Categoricals are answered from their categories - fillna() on one raises rather than
    # reporting emptiness, and by Step 12 the text columns are categorical.
    if isinstance(s.dtype, pd.CategoricalDtype):
        return all(str(v).strip() == '' for v in s.cat.categories)
    # pandas 3 gives string columns a `str` dtype; pandas 2 leaves them `object`. Anything else
    # (numeric, bool, datetime) has no notion of blank.
    if not (s.dtype == object or pd.api.types.is_string_dtype(s)):
        return False
    return s.astype('object').fillna('').astype(str).str.strip().eq('').all()


empty_cols    = [c for c in df.columns if df[c].isna().all() or all_blank(df[c])]
constant_cols = [c for c in df.columns if c not in empty_cols and df[c].nunique(dropna=True) <= 1]
const_vals    = {c: df[c].dropna().iloc[0] if df[c].notna().any() else None for c in constant_cols}

print('empty    (all NaN or blank) :', empty_cols)
print('constant (1 value only)     :', {c: str(v) for c, v in const_vals.items()})

M['dead_cols'] = {'empty': empty_cols, 'constant': {c: str(v) for c, v in const_vals.items()}}

before = snap(df)
df = df.drop(columns=empty_cols + constant_cols)
compare(before, df, 'Step 4 - drop empty and zero-variance columns')

empty    (all NaN or blank) : ['occupationId']
constant (1 value only)     : {'salary_type': 'Monthly', 'status_id': '0'}
=== Step 4 - drop empty and zero-variance columns ===
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     22     409.4    1,044,587
AFTER     1,044,587     19     376.9            0
DELTA            +0     -3     -32.5   -1,044,587
    columns removed: ['occupationId', 'salary_type', 'status_id']



{'rows': 1044587,
 'cols': 19,
 'mem': np.float64(376.9),
 'na': 0,
 'columns': ['categories',
  'employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary']}

## 5 · Type conversion — dates

`object`/`str` → `datetime64[ns]`. **Loss check:** the conversion is only accepted if
`to_datetime(...).dt.strftime('%Y-%m-%d')` reproduces the original string on every row. If that
assertion holds, the change is provably lossless.

Cross-field logic is validated at the same time — a repost cannot predate its original, and a
listing cannot expire before it is posted.

In [7]:
DATE_COLS = ['metadata_newPostingDate', 'metadata_originalPostingDate', 'metadata_expiryDate']

print('BEFORE')
print(df[DATE_COLS].dtypes.astype(str).to_string())
print(df[DATE_COLS].head(3).to_string(index=False), '\n')

date_rows = []
before = snap(df)
for c in DATE_COLS:
    original = df[c]
    parsed   = pd.to_datetime(original, errors='coerce')
    unparsed = int(parsed.isna().sum() - original.isna().sum())
    lossless = bool((parsed.dt.strftime('%Y-%m-%d') == original).all())
    assert unparsed == 0 and lossless, f'{c} would lose data - refusing to convert'
    df[c] = parsed
    date_rows.append({'column': c, 'from': str(original.dtype), 'to': str(parsed.dtype),
                      'unparseable': unparsed, 'round_trip_identical': lossless,
                      'min': str(parsed.min().date()), 'max': str(parsed.max().date())})

date_tbl = pd.DataFrame(date_rows)
M['md_dates'] = to_md(date_tbl, index=False)

d_new, d_orig, d_exp = df[DATE_COLS[0]], df[DATE_COLS[1]], df[DATE_COLS[2]]
M['date_checks'] = {'orig_after_new': int((d_orig > d_new).sum()),
                    'expiry_before_post': int((d_exp <= d_new).sum()),
                    'lifespan_median': float((d_exp - d_new).dt.days.median()),
                    'lifespan_max': int((d_exp - d_new).dt.days.max())}

print('AFTER')
print(df[DATE_COLS].dtypes.astype(str).to_string())
print(date_tbl.to_string(index=False), '\n')
print('originalPostingDate > newPostingDate :', M['date_checks']['orig_after_new'], '(violations)')
print('expiryDate <= newPostingDate         :', M['date_checks']['expiry_before_post'], '(violations)')
print('listing lifespan: median', M['date_checks']['lifespan_median'], 'days, max',
      M['date_checks']['lifespan_max'], 'days\n')
M['date_dtype'] = str(df[DATE_COLS[0]].dtype)
compare(before, df, 'Step 5 - parse date columns',
        f'str -> {M["date_dtype"]}, round-trip verified')

BEFORE
metadata_newPostingDate         str
metadata_originalPostingDate    str
metadata_expiryDate             str
metadata_newPostingDate metadata_originalPostingDate metadata_expiryDate
             2023-04-08                   2023-03-30          2023-05-08
             2023-04-08                   2023-04-08          2023-05-08
             2023-04-08                   2023-04-08          2023-04-22 



AFTER
metadata_newPostingDate         datetime64[us]
metadata_originalPostingDate    datetime64[us]
metadata_expiryDate             datetime64[us]
                      column from             to  unparseable  round_trip_identical        min        max
     metadata_newPostingDate  str datetime64[us]            0                  True 2023-03-28 2024-05-29
metadata_originalPostingDate  str datetime64[us]            0                  True 2022-10-03 2024-05-29
         metadata_expiryDate  str datetime64[us]            0                  True 2023-04-04 2024-06-28 

originalPostingDate > newPostingDate : 0 (violations)
expiryDate <= newPostingDate         : 0 (violations)
listing lifespan: median 30.0 days, max 30 days

=== Step 5 - parse date columns ===
    str -> datetime64[us], round-trip verified
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     19     376.9            0
AFTER     1,044,587     19     345.2            0
DELTA            +0     +0     -31.7 

{'rows': 1044587,
 'cols': 19,
 'mem': np.float64(345.2),
 'na': 0,
 'columns': ['categories',
  'employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary']}

## 6 · Type conversion — `categories` JSON → bridge table

`categories` holds a JSON array of `{"id", "category"}` objects. As a string it is unqueryable:
filtering "all IT jobs" needs a substring match that also catches the label inside other text.

The **many-to-many** structure is preserved in a separate `job_category` bridge table.
Keeping only the first category would be the lossy shortcut here — the notebook measures exactly
how many category assignments that would discard before deciding.

In [8]:
before = snap(df)
parsed_cat = df['categories'].map(json.loads)
n_per_row  = parsed_cat.map(len)

job_category = (pd.DataFrame({'metadata_jobPostId': df['metadata_jobPostId'], 'c': parsed_cat})
                  .explode('c', ignore_index=True))
job_category['category_id'] = job_category['c'].map(lambda d: d['id']).astype('int16')
job_category['category']    = job_category['c'].map(lambda d: d['category']).astype('category')
job_category = job_category.drop(columns='c')

df['primary_category'] = parsed_cat.map(lambda v: v[0]['category'])
df['n_categories']     = n_per_row.astype('int8')

M['categories'] = {
    'distinct': int(job_category['category'].nunique()),
    'assignments': len(job_category),
    'empty_arrays': int((n_per_row == 0).sum()),
    'per_row': {int(k): int(v) for k, v in n_per_row.value_counts().sort_index().items()},
    'multi_pct': round((n_per_row > 1).mean() * 100, 1),
    'lost_if_first_only': int(len(job_category) - len(df)),
}
M['md_top_categories'] = to_md(
    job_category['category'].value_counts().head(10).rename('postings').to_frame())

print('categories per posting :', M['categories']['per_row'])
print('distinct categories    :', M['categories']['distinct'])
print('total assignments      :', f'{M["categories"]["assignments"]:,}')
print(f'multi-category rows    : {M["categories"]["multi_pct"]}%  '
      f'-> keeping only the first would discard {M["categories"]["lost_if_first_only"]:,} assignments\n')
print('bridge table:')
print(job_category.head(4).to_string(index=False), '\n')

df = df.drop(columns='categories')
compare(before, df, 'Step 6 - normalise categories JSON',
        f'{len(job_category):,}-row bridge table extracted; raw JSON column dropped')

categories per posting : {1: 654951, 2: 208746, 3: 85461, 4: 38186, 5: 57243}
distinct categories    : 43
total assignments      : 1,767,785
multi-category rows    : 37.3%  -> keeping only the first would discard 723,198 assignments

bridge table:
metadata_jobPostId  category_id                    category
  MCF-2023-0252866           13        Environment / Health
  MCF-2023-0252866           25               Manufacturing
  MCF-2023-0252866           36 Sciences / Laboratory / R&D
  MCF-2023-0273977           21      Information Technology 

=== Step 6 - normalise categories JSON ===
    1,767,785-row bridge table extracted; raw JSON column dropped
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     19     345.2            0
AFTER     1,044,587     20     291.4            0
DELTA            +0     +1     -53.8           +0
    columns removed: ['categories']
    columns added  : ['primary_category', 'n_categories']



{'rows': 1044587,
 'cols': 20,
 'mem': np.float64(291.4),
 'na': 0,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories']}

## 7 · Data fixing — salaries

The floor and ceiling are **not treated symmetrically**, and that asymmetry is deliberate:

* **`SALARY_FLOOR` fixes corrupted data.** `$1/month` is not a low wage, it is a required-field
  placeholder, and `$10`–`$15` is an hourly rate typed into a monthly-only field. Neither is a
  real number at *any* resolution, so there is nothing to preserve — both bounds are **nulled**.
* **`SALARY_CEILING` is a statistical judgment, not a data-quality fix.** Whether a $150,000/month
  posting is a genuine C-suite role or a missing decimal depends on the question being asked — a
  mean/std view wants it excluded, a fraud-detection or "highest paid roles" view wants to see it.
  That decision belongs to analysis, not to this pipeline, so ceiling violations are **flagged, not
  nulled**: the raw values are left untouched and a `salary_flag == 'outlier'` marker is added so
  any downstream aggregate can choose to filter them out with one line, or choose not to.

This mirrors the general rule: cap what is definitely wrong at cleaning time; flag what is
*probably* wrong and let the analysis stage decide, since the right threshold there depends on
what's being measured.

Floor values are **nulled, not winsorised**: winsorising invents a number, `NaN` admits ignorance
and lets `mean()` skip the row. A quoted salary is a **range**, so the pair is the unit of
validity for the floor check — if either bound is a placeholder, both are nulled. Nulling only the
maximum would leave rows like `$1 – $600` behind, whose average of `$300.50` sits below the floor
the rule is supposed to enforce.

The plain 1.5×IQR rule is explicitly rejected below regardless of which stage would apply it — it
would delete tens of thousands of legitimate senior roles.

**One carve-out on the floor side.** `SALARY_FLOOR` was checked against the risk of nulling
genuine internship stipends before being set: internship salaries in this file have a median of
$1,200 and a 5th percentile of $800, so almost none are legitimately below $500 — but the rare ones
that are shouldn't be silently destroyed. Rows tagged `Internship/Attachment` with both bounds in
`[INTERN_STIPEND_FLOOR, SALARY_FLOOR)` are **kept, not nulled**, and marked `low_stipend` rather
than `undisclosed`.

Requires nullable `Int32`, since a plain `int64` column cannot hold `NaN`.

In [9]:
before = snap(df)
sal_before = df[['salary_minimum', 'salary_maximum', 'average_salary']].describe(
    percentiles=[.01, .5, .99, .999]).round(1)
print('BEFORE\n', sal_before.to_string(), '\n')

# why not IQR: show what the textbook rule would have cost
q1, q3 = df['average_salary'].quantile([.25, .75])
iqr_fence = q3 + 3 * (q3 - q1)
M['salary_iqr'] = {'q1': float(q1), 'q3': float(q3), 'fence_3iqr': float(iqr_fence),
                   'rows_above_fence': int((df['average_salary'] > iqr_fence).sum())}
print(f'3xIQR upper fence would be ${iqr_fence:,.0f} and would null '
      f'{M["salary_iqr"]["rows_above_fence"]:,} rows - rejected as far too aggressive\n')

# float32 viability is measured BEFORE the fix as well as after (see Step 11) - the answer changes
M['float32_before'] = {
    'max_error': float((df['average_salary'].astype('float32').astype('float64')
                        - df['average_salary']).abs().max()),
    'max_value': float(df['average_salary'].max()),
}
print(f'float32 round-trip error on average_salary as-is: '
      f'{M["float32_before"]["max_error"]} (max value ${M["float32_before"]["max_value"]:,.0f})\n')

df['salary_minimum'] = df['salary_minimum'].astype('Int32')
df['salary_maximum'] = df['salary_maximum'].astype('Int32')

# Floor and ceiling are handled asymmetrically on purpose (see markdown above):
# the floor nulls corrupted data, the ceiling only flags a statistical judgment call.
low_min    = df['salary_minimum'] < SALARY_FLOOR
low_max    = df['salary_maximum'] < SALARY_FLOOR
high_max   = df['salary_maximum'] > SALARY_CEILING
sentinel_1 = int(((df['salary_minimum'] == 1) & (df['salary_maximum'] == 1)).sum())
inverted   = int((df['salary_minimum'] > df['salary_maximum']).sum())

# internship carve-out: a low bound is a plausible stipend, not a placeholder, if BOTH bounds
# sit in [INTERN_STIPEND_FLOOR, SALARY_FLOOR) on a row tagged Internship/Attachment
is_internship  = df['employmentTypes'] == 'Internship/Attachment'
intern_stipend = (is_internship
                  & low_min & low_max
                  & (df['salary_minimum'] >= INTERN_STIPEND_FLOOR)
                  & (df['salary_maximum'] >= INTERN_STIPEND_FLOOR))

# only the floor defect is nulled - the ceiling defect is left in place and flagged instead
bad = (low_min | low_max) & ~intern_stipend

M['salary_fix'] = {
    'floor': SALARY_FLOOR, 'ceiling': SALARY_CEILING,
    'intern_stipend_floor': INTERN_STIPEND_FLOOR,
    'low_min_n': int(low_min.sum()), 'low_max_n': int(low_max.sum()),
    'high_n': int(high_max.sum()),
    'min_only_n': int((low_min & ~low_max).sum()),
    'intern_stipend_n': int(intern_stipend.sum()),
    'sentinel_exactly_1': sentinel_1, 'inverted': inverted,
    'max_before': int(sal_before.loc['max', 'salary_maximum']),
    'low_by_employment': df.loc[low_min | low_max, 'employmentTypes'].value_counts().head(4).to_dict(),
}
print(f'salary_minimum < {SALARY_FLOOR:,}   : {M["salary_fix"]["low_min_n"]:,} rows  (nulled)')
print(f'salary_maximum < {SALARY_FLOOR:,}   : {M["salary_fix"]["low_max_n"]:,} rows  (nulled, '
      f'of which exactly $1-$1: {sentinel_1:,})')
print(f'   -> low minimum but a plausible maximum: {M["salary_fix"]["min_only_n"]:,} rows, '
      f'caught only by the pair rule')
print(f'salary_maximum > {SALARY_CEILING:,} : {M["salary_fix"]["high_n"]:,} rows  '
      f'(kept, flagged "outlier" - NOT nulled)')
print(f'internship stipend carve-out ({INTERN_STIPEND_FLOOR:,}-{SALARY_FLOOR:,}): '
      f'{M["salary_fix"]["intern_stipend_n"]:,} rows kept, flagged low_stipend')
print('salary_minimum > salary_maximum :', inverted, '(no swap correction needed)')
print('employment mix of the low group :', M['salary_fix']['low_by_employment'], '\n')

df.loc[bad, ['salary_minimum', 'salary_maximum']] = pd.NA
df['salary_na']   = df['salary_maximum'].isna()
df['salary_flag'] = pd.Series(np.select(
    [bad,           intern_stipend, high_max],
    ['undisclosed', 'low_stipend',  'outlier'],
    default='ok',
), index=df.index).astype('category')
df['average_salary'] = ((df['salary_minimum'] + df['salary_maximum']) / 2).astype('Float64')

sal_after = df[['salary_minimum', 'salary_maximum', 'average_salary']].describe(
    percentiles=[.01, .5, .99, .999]).round(1)
M['md_salary_before'] = to_md(sal_before)
M['md_salary_after']  = to_md(sal_after)
M['salary_fix']['nulled'] = int(bad.sum())
M['salary_fix']['coverage_pct'] = round(df['salary_maximum'].notna().mean() * 100, 2)
M['salary_fix']['flag_counts'] = df['salary_flag'].value_counts().to_dict()

print('AFTER\n', sal_after.to_string(), '\n')
print(f'salaries nulled: {int(bad.sum()):,}  |  remaining coverage: '
      f'{M["salary_fix"]["coverage_pct"]}%')
print('salary_flag breakdown:', M['salary_fix']['flag_counts'])
compare(before, df, 'Step 7 - fix salary sentinels and outliers',
        f'{int(bad.sum()):,} salaries -> NaN; {M["salary_fix"]["intern_stipend_n"]} internship '
        f'stipends kept; average_salary recomputed; salary_na/salary_flag built')

BEFORE
        salary_minimum  salary_maximum  average_salary
count       1044587.0       1044587.0       1044587.0
mean           3828.2          5630.8          4729.5
std            3104.3         27096.0         13882.9
min               1.0             1.0             1.0
1%              500.0          1000.0           800.0
50%            3000.0          4500.0          3800.0
99%           13000.0         20000.0         16750.0
99.9%         20000.0         35000.0         27470.7
max          350000.0      25330000.0      12666400.0 

3xIQR upper fence would be $13,300 and would null 24,520 rows - rejected as far too aggressive

float32 round-trip error on average_salary as-is: 0.0 (max value $12,666,400)

salary_minimum < 500   : 9,581 rows  (nulled)
salary_maximum < 500   : 7,124 rows  (nulled, of which exactly $1-$1: 1,804)
   -> low minimum but a plausible maximum: 2,457 rows, caught only by the pair rule
salary_maximum > 100,000 : 269 rows  (kept, flagged "outlier" - NOT 

AFTER
        salary_minimum  salary_maximum  average_salary
count       1035010.0       1035010.0       1035010.0
mean           3862.9          5673.5          4768.2
std            3097.4         27193.5         13929.4
min             300.0           400.0           350.0
1%             1000.0          1400.0          1150.0
50%            3000.0          4500.0          3850.0
99%           13000.0         20000.0         16875.0
99.9%         20000.0         35000.0         27249.1
max          350000.0      25330000.0      12666400.0 

salaries nulled: 9,577  |  remaining coverage: 99.08%
salary_flag breakdown: {'ok': 1034742, 'undisclosed': 9577, 'outlier': 264, 'low_stipend': 4}
=== Step 7 - fix salary sentinels and outliers ===
    9,577 salaries -> NaN; 4 internship stipends kept; average_salary recomputed; salary_na/salary_flag built
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     20     291.4            0
AFTER     1,044,587     22     288.3      

{'rows': 1044587,
 'cols': 22,
 'mem': np.float64(288.3),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_na',
  'salary_flag']}

### `minimumYearsExperience`

Range check only. Values above `EXPERIENCE_MAX` are not physically possible, but they are a
handful of rows and are a poster-side typo rather than a pipeline defect — nulling them changes no
aggregate. **`0` is left untouched: it is a real value** meaning "no experience required".

In [10]:
before = snap(df)
exp = df['minimumYearsExperience']
impossible = exp > EXPERIENCE_MAX
M['experience'] = {'max_before': int(exp.max()), 'impossible': int(impossible.sum()),
                   'zeros': int((exp == 0).sum()), 'zeros_pct': round((exp == 0).mean() * 100, 1),
                   'tail': exp[exp > 20].value_counts().sort_index().tail(6).to_dict()}
print('BEFORE  max =', M['experience']['max_before'], '| tail values:', M['experience']['tail'])

df['minimumYearsExperience'] = exp.astype('Int16').mask(impossible, pd.NA)
print('AFTER   max =', int(df['minimumYearsExperience'].max()),
      f'| {M["experience"]["impossible"]} impossible values -> NaN')
compare(before, df, 'Step 7b - cap impossible experience values')

BEFORE  max = 88 | tail values: {59: 1, 61: 1, 62: 1, 76: 1, 87: 2, 88: 1}
AFTER   max = 88 | 0 impossible values -> NaN
=== Step 7b - cap impossible experience values ===
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     22     288.3       28,731
AFTER     1,044,587     22     283.1       28,731
DELTA            +0     +0      -5.2           +0



{'rows': 1044587,
 'cols': 22,
 'mem': np.float64(283.1),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_na',
  'salary_flag']}

## 8 · Data fixing — text normalisation

Raw display strings are preserved; normalised join keys are added alongside. Case-folding alone
collapses thousands of distinct titles — without it, `Software Engineer` and `software engineer`
are counted as different roles in every top-titles chart.

In [11]:
before = snap(df)
t_raw, c_raw = df['title'], df['postedCompany_name']

M['text'] = {
    'title_ws_rows': int((t_raw != t_raw.str.strip()).sum()),
    'title_distinct_raw': int(t_raw.nunique()),
    'company_ws_rows': int((c_raw != c_raw.str.strip()).sum()),
    'company_distinct_raw': int(c_raw.nunique()),
}

df['title'] = t_raw.str.strip().str.replace(r'\s+', ' ', regex=True)
df['title_normalised']   = df['title'].str.lower()
df['company_normalised'] = c_raw.str.upper().str.replace(r'\s+', ' ', regex=True).str.strip()

M['text'].update({
    'title_distinct_clean': int(df['title'].nunique()),
    'title_distinct_normalised': int(df['title_normalised'].nunique()),
    'company_distinct_normalised': int(df['company_normalised'].nunique()),
})
M['text']['title_collapsed'] = M['text']['title_distinct_raw'] - M['text']['title_distinct_normalised']

print(f'title    : {M["text"]["title_ws_rows"]:,} rows had stray whitespace')
print(f'           distinct  raw {M["text"]["title_distinct_raw"]:,}'
      f'  -> stripped {M["text"]["title_distinct_clean"]:,}'
      f'  -> lowercased {M["text"]["title_distinct_normalised"]:,}'
      f'   ({M["text"]["title_collapsed"]:,} collapsed)')
print(f'company  : distinct raw {M["text"]["company_distinct_raw"]:,}'
      f'  -> normalised {M["text"]["company_distinct_normalised"]:,}\n')
compare(before, df, 'Step 8 - normalise title and company text')

title    : 4,445 rows had stray whitespace
           distinct  raw 377,084  -> stripped 375,517  -> lowercased 364,760   (12,324 collapsed)
company  : distinct raw 53,151  -> normalised 53,150

=== Step 8 - normalise title and company text ===
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     22     283.1       28,731
AFTER     1,044,587     24     366.3       28,731
DELTA            +0     +2     +83.2           +0
    columns added  : ['title_normalised', 'company_normalised']



{'rows': 1044587,
 'cols': 24,
 'mem': np.float64(366.3),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_na',
  'salary_flag',
  'title_normalised',
  'company_normalised']}

## 9 · Data filling

**Almost nothing is filled, on purpose.** After the ghost rows are gone there are no explicit
`NaN`s left except the salaries this notebook deliberately created. The real question is the
*implicit* missingness encoded as `0`, and for each such column the answer is to leave it and
document it:

| column | zeros | decision |
|---|---|---|
| `metadata_totalNumberJobApplication` | ~63% | leave — genuinely ambiguous, cannot separate "no applicants" from "not tracked" |
| `metadata_totalNumberOfView` | ~17% | leave — a 0-view / 0-application posting is internally consistent |
| `minimumYearsExperience` | ~11% | leave — `0` is a real value for entry-level roles |
| `salary_*` | nulled above | **never** median-impute; salary is the dependent variable |

Filling salaries by group median would manufacture the very distribution the dashboard exists to
measure, and would tighten variance so every confidence interval comes out wrong. Coverage is
reported alongside instead.

The zero-inflation claim for `minimumYearsExperience` is *tested* below, not asserted: if the
zeros were spread evenly across seniority they would be a default value, not a real one.

In [12]:
before = snap(df)
zero_cols = ['metadata_totalNumberJobApplication', 'metadata_totalNumberOfView',
             'minimumYearsExperience', 'numberOfVacancies', 'metadata_repostCount']
zero_tbl = pd.DataFrame({
    'zeros':   [int((df[c] == 0).sum()) for c in zero_cols],
    'zero_pct': [round((df[c] == 0).mean() * 100, 1) for c in zero_cols],
    'decision': ['leave - ambiguous', 'leave - ambiguous', 'leave - real value',
                 'n/a - min is 1', 'leave - real value'],
}, index=zero_cols)
print(zero_tbl.to_string(), '\n')
M['md_zeros'] = to_md(zero_tbl)

# test the "0 years means entry level" claim instead of assuming it
zero_exp_mix = (df.loc[df['minimumYearsExperience'] == 0, 'positionLevels']
                  .value_counts(normalize=True).mul(100).round(1).head(5))
overall_mix  = df['positionLevels'].value_counts(normalize=True).mul(100).round(1)
mix = pd.DataFrame({'zero_exp_%': zero_exp_mix,
                    'overall_%': overall_mix.reindex(zero_exp_mix.index)})
print('seniority mix of rows with 0 years required, vs the frame overall:')
print(mix.to_string(), '\n')
M['md_zero_exp_mix'] = to_md(mix)

# derived columns - added rather than filled
df['listing_days'] = (df['metadata_expiryDate'] - df['metadata_newPostingDate']).dt.days.astype('int16')
df['is_repost']    = df['metadata_repostCount'] > 0
df['source']       = df['metadata_jobPostId'].str.extract(r'^([A-Za-z]+)')[0]

M['derived'] = {'listing_days': 'expiryDate - newPostingDate',
                'is_repost': 'repostCount > 0',
                'source': 'ID prefix (MCF / ATS)',
                'salary_na': 'salary_maximum.isna() after Step 7',
                'salary_flag': "'ok' / 'undisclosed' / 'outlier' / 'low_stipend' - reason code from Step 7; "
                               "'outlier' rows keep their raw value (not nulled)",
                'average_salary': '(min + max) / 2, recomputed after Step 7',
                'primary_category': 'first element of the categories JSON',
                'n_categories': 'length of the categories JSON'}
M['source_mix'] = df['source'].value_counts().to_dict()
M['filled_cells'] = 0
print('source mix:', M['source_mix'])
compare(before, df, 'Step 9 - filling decisions and derived columns',
        '0 cells imputed; 3 derived columns added')

                                      zeros  zero_pct            decision
metadata_totalNumberJobApplication   656375      62.8   leave - ambiguous
metadata_totalNumberOfView           179121      17.1   leave - ambiguous
minimumYearsExperience               114451      11.0  leave - real value
numberOfVacancies                         0       0.0      n/a - min is 1
metadata_repostCount                1001862      95.9  leave - real value 

seniority mix of rows with 0 years required, vs the frame overall:
                   zero_exp_%  overall_%
positionLevels                          
Fresh/entry level        61.7       11.4
Non-executive            13.6       12.6
Executive                10.0       24.3
Junior Executive          8.2       16.0
Professional              4.5       10.7 



source mix: {'MCF': 1044463, 'ATS': 124}
=== Step 9 - filling decisions and derived columns ===
    0 cells imputed; 3 derived columns added
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     24     366.3       28,731
AFTER     1,044,587     27     381.0       28,731
DELTA            +0     +3     +14.7           +0
    columns added  : ['listing_days', 'is_repost', 'source']



{'rows': 1044587,
 'cols': 27,
 'mem': np.float64(381.0),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_na',
  'salary_flag',
  'title_normalised',
  'company_normalised',
  'listing_days',
  'is_repost',
  'source']}

## 10 · Duplicate rows

Duplication is tested at five levels of strictness, because "duplicate" means different things
depending on the question being asked.

The decisive evidence is at the bottom: for same-day identical postings, most groups have
**different view counts**. That means the platform served them as separate listings that
accumulated separate traffic — they are real distinct records, not a loading artefact. So they are
**flagged, not dropped**, and the choice is left to the analysis that consumes the data.

In [13]:
before = snap(df)
CONTENT = ['title_normalised', 'company_normalised', 'employmentTypes', 'positionLevels',
           'primary_category', 'salary_minimum', 'salary_maximum', 'numberOfVacancies']
KEYS = {
    '1. exact duplicate row (all columns)':        list(df.columns),
    '2. duplicate primary key (jobPostId)':        ['metadata_jobPostId'],
    '3. identical content, any date':              CONTENT,
    '4. identical content, same posting date':     CONTENT + ['metadata_newPostingDate'],
    '5. company + title + posting date':           ['company_normalised', 'title_normalised',
                                                    'metadata_newPostingDate'],
}
dup_tbl = pd.DataFrame(
    [{'key': k, 'duplicate_rows': int(df.duplicated(subset=v).sum()),
      'pct': round(df.duplicated(subset=v).mean() * 100, 2)} for k, v in KEYS.items()]
).set_index('key')
print(dup_tbl.to_string(), '\n')
M['md_duplicates'] = to_md(dup_tbl)
M['dup'] = {'exact': int(dup_tbl.iloc[0, 0]), 'pk': int(dup_tbl.iloc[1, 0]),
            'content': int(dup_tbl.iloc[2, 0]), 'same_day': int(dup_tbl.iloc[3, 0])}

SAME_DAY = CONTENT + ['metadata_newPostingDate']
grp = df.groupby(SAME_DAY, dropna=False, observed=True)
df['dup_group_size'] = grp['title'].transform('size').astype('int16')
df['is_same_day_dup'] = df['dup_group_size'] > 1

d = df[df['is_same_day_dup']]
nun = d.groupby(SAME_DAY, dropna=False, observed=True)[
    ['metadata_totalNumberOfView', 'metadata_totalNumberJobApplication']].nunique()
M['dup'].update({
    'same_day_rows': int(len(d)),
    'same_day_groups': int(len(nun)),
    'groups_views_differ': int((nun['metadata_totalNumberOfView'] > 1).sum()),
    'groups_apps_differ': int((nun['metadata_totalNumberJobApplication'] > 1).sum()),
    'largest_group': int(df['dup_group_size'].max()),
    'behalf_in_dups': round(d['metadata_isPostedOnBehalf'].mean() * 100, 1),
    'behalf_baseline': round(df['metadata_isPostedOnBehalf'].mean() * 100, 1),
})
M['md_dup_agencies'] = to_md(
    d['postedCompany_name'].value_counts().head(5).rename('duplicate_rows').to_frame())

print(f'same-day duplicate rows      : {M["dup"]["same_day_rows"]:,} in '
      f'{M["dup"]["same_day_groups"]:,} groups (largest {M["dup"]["largest_group"]:,})')
print(f'groups where views differ    : {M["dup"]["groups_views_differ"]:,} / '
      f'{M["dup"]["same_day_groups"]:,}  <- proof these are distinct listings')
print(f'groups where apps differ     : {M["dup"]["groups_apps_differ"]:,}')
print(f'posted-on-behalf in dup rows : {M["dup"]["behalf_in_dups"]}% vs '
      f'{M["dup"]["behalf_baseline"]}% baseline  <- recruitment agencies')
print('\ntop agencies by duplicate rows:')
print(d['postedCompany_name'].value_counts().head(5).to_string(), '\n')
compare(before, df, 'Step 10 - flag duplicates',
        '0 rows dropped; dup_group_size + is_same_day_dup added')

                                         duplicate_rows    pct
key                                                           
1. exact duplicate row (all columns)                  0   0.00
2. duplicate primary key (jobPostId)                  0   0.00
3. identical content, any date                   369888  35.41
4. identical content, same posting date           34878   3.34
5. company + title + posting date                 49619   4.75 



same-day duplicate rows      : 56,222 in 21,344 groups (largest 260)
groups where views differ    : 16,990 / 21,344  <- proof these are distinct listings
groups where apps differ     : 5,902
posted-on-behalf in dup rows : 37.8% vs 5.9% baseline  <- recruitment agencies

top agencies by duplicate rows:
postedCompany_name
RECRUITPEDIA PTE. LTD.               16044
HD MANPOWER CONSULTANTS PTE. LTD.     3519
57 EMPLOYMENT AGENCY PTE. LTD.        2977
MINDFLEX EDUCATION PTE. LTD.          2814
ORIENTAL EMPLOYMENT PTE. LTD.         2394 

=== Step 10 - flag duplicates ===
    0 rows dropped; dup_group_size + is_same_day_dup added
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     27     381.0       28,731
AFTER     1,044,587     29     384.1       28,731
DELTA            +0     +2      +3.1           +0
    columns added  : ['dup_group_size', 'is_same_day_dup']



{'rows': 1044587,
 'cols': 29,
 'mem': np.float64(384.1),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_na',
  'salary_flag',
  'title_normalised',
  'company_normalised',
  'listing_days',
  'is_repost',
  'source',
  'dup_group_size',
  'is_same_day_dup']}

## 11 · Type conversion — categorical and downcast

Left until last, so the category sets contain only surviving values and the integer ranges are
measured on cleaned data.

Every downcast is **asserted safe** against the observed range before it is applied — the loop
refuses rather than silently wrapping. `metadata_totalNumberOfView` and
`metadata_totalNumberJobApplication` get `int32` rather than the tighter `int16` their current
range allows: `int16` leaves only ~24k headroom, and a single viral posting in a future extract
would overflow it silently.

`average_salary` is the interesting case, and it shows why dtype decisions belong at the *end* of a
pipeline: whether `float32` is lossy depends on how much cleaning has already happened. The cell
below measures the round-trip error at three points and reports all of them rather than asserting
a verdict.

In [14]:
before = snap(df)
DOWNCAST = {'minimumYearsExperience': 'Int8', 'metadata_repostCount': 'int8',
            'numberOfVacancies': 'int16', 'metadata_totalNumberOfView': 'int32',
            'metadata_totalNumberJobApplication': 'int32', 'listing_days': 'int16',
            'n_categories': 'int8'}
CATEGORICAL = ['employmentTypes', 'positionLevels', 'status_jobStatus',
               'postedCompany_name', 'company_normalised', 'primary_category', 'source']

rows = []
for c, t in DOWNCAST.items():
    lo, hi = df[c].min(), df[c].max()
    info = np.iinfo(np.dtype(t.lower()))
    safe = bool(lo >= info.min and hi <= info.max)
    assert safe, f'{c} range [{lo},{hi}] does not fit {t}'
    rows.append({'column': c, 'from': str(df[c].dtype), 'to': t,
                 'observed_range': f'{lo} - {hi}', 'headroom': f'{info.max - hi:,}', 'safe': safe})
    df[c] = df[c].astype(t)

for c in CATEGORICAL:
    n = df[c].nunique()
    mem_b = df[c].memory_usage(deep=True) / 1e6
    df[c] = df[c].astype('category')
    rows.append({'column': c, 'from': 'str', 'to': 'category',
                 'observed_range': f'{n:,} distinct',
                 'headroom': f'{mem_b - df[c].memory_usage(deep=True)/1e6:,.1f} MB saved', 'safe': True})

type_tbl = pd.DataFrame(rows)
M['md_types'] = to_md(type_tbl, index=False)
print(type_tbl.to_string(index=False), '\n')

# float32 viability, measured rather than assumed - compare against the pre-fix answer
f32_err = float((df['average_salary'].astype('Float32').astype('Float64')
                 - df['average_salary']).abs().max())
half_vals = int((df['average_salary'].dropna() % 1 != 0).sum())
M['float32'] = {'max_error': f32_err, 'half_values': half_vals,
                'max_value': float(df['average_salary'].max())}
print('float32 round-trip error on average_salary, at three points in the pipeline:')
print(f'   raw file                    : {M["float32_raw"]["max_error"]}  '
      f'(max ${M["float32_raw"]["max_value"]:,.0f})')
print(f'   after synthetic rows removed: {M["float32_before"]["max_error"]}  '
      f'(max ${M["float32_before"]["max_value"]:,.0f})')
print(f'   after salary fix            : {f32_err}  (max ${M["float32"]["max_value"]:,.0f})')
print(f'   {half_vals:,} rows carry a genuine .5 fraction')
print('   -> kept as Float64 anyway: the saving is only a few MB and float32 would silently')
print('      re-break if a future extract reintroduces large values\n')

compare(before, df, 'Step 11 - categorical dtypes and integer downcast')

                            column  from       to  observed_range      headroom  safe
            minimumYearsExperience Int16     Int8          0 - 88            39  True
              metadata_repostCount int64     int8           0 - 2           125  True
                 numberOfVacancies int64    int16         1 - 999        31,768  True
        metadata_totalNumberOfView int64    int32        0 - 8190 2,147,475,457  True
metadata_totalNumberJobApplication int64    int32        0 - 1342 2,147,482,305  True
                      listing_days int16    int16          1 - 30        32,737  True
                      n_categories  int8     int8           1 - 5           122  True
                   employmentTypes   str category      8 distinct 16.8 MB saved  True
                    positionLevels   str category      9 distinct 20.7 MB saved  True
                  status_jobStatus   str category      3 distinct 11.9 MB saved  True
                postedCompany_name   str category 53,1

{'rows': 1044587,
 'cols': 29,
 'mem': np.float64(211.4),
 'na': 28731,
 'columns': ['employmentTypes',
  'metadata_expiryDate',
  'metadata_isPostedOnBehalf',
  'metadata_jobPostId',
  'metadata_newPostingDate',
  'metadata_originalPostingDate',
  'metadata_repostCount',
  'metadata_totalNumberJobApplication',
  'metadata_totalNumberOfView',
  'minimumYearsExperience',
  'numberOfVacancies',
  'positionLevels',
  'postedCompany_name',
  'salary_maximum',
  'salary_minimum',
  'status_jobStatus',
  'title',
  'average_salary',
  'primary_category',
  'n_categories',
  'salary_na',
  'salary_flag',
  'title_normalised',
  'company_normalised',
  'listing_days',
  'is_repost',
  'source',
  'dup_group_size',
  'is_same_day_dup']}

## 12 · Final validation and save

In [15]:
final = profile(df)
print(final.to_string(), '\n')
M['md_profile_final'] = to_md(final)
M['final'] = snap(df)

assert df['metadata_jobPostId'].is_unique,     'primary key is not unique'
assert df['title'].notna().all(),              'ghost rows survived'
assert not df['metadata_jobPostId'].str.match(SYNTHETIC_ID_RE).any(), 'synthetic rows survived'
ok_rows      = df['salary_flag'] == 'ok'
stipend_rows = df['salary_flag'] == 'low_stipend'
outlier_rows = df['salary_flag'] == 'outlier'
undisclosed_rows = df['salary_flag'] == 'undisclosed'
assert df.loc[ok_rows, 'salary_maximum'].between(SALARY_FLOOR, SALARY_CEILING).all(), 'salary max out of range'
assert df.loc[ok_rows, 'salary_minimum'].between(SALARY_FLOOR, SALARY_CEILING).all(), 'salary min out of range'
assert df.loc[stipend_rows, 'salary_maximum'].between(INTERN_STIPEND_FLOOR, SALARY_FLOOR).all(), 'stipend carve-out out of range'
assert (df.loc[stipend_rows, 'employmentTypes'] == 'Internship/Attachment').all(), 'stipend carve-out leaked outside internships'
assert (df.loc[outlier_rows, 'salary_maximum'] > SALARY_CEILING).all(), 'outlier flag set on a row that is not actually over the ceiling'
assert df.loc[outlier_rows, 'salary_maximum'].notna().all(), 'outlier rows were nulled instead of flagged'
assert undisclosed_rows.sum() == int(df['salary_maximum'].isna().sum()), 'nulled rows and undisclosed flag disagree'
assert (df['salary_minimum'].isna() == df['salary_maximum'].isna()).all(), 'salary bounds nulled apart'
assert (df['metadata_expiryDate'] > df['metadata_newPostingDate']).all(), 'date logic violated'

# No dead column may reach jobs_clean.parquet - neither carried through from the source nor
# reintroduced by a derived column. Checked by name, and by re-running the Step 4 test.
leaked = [c for c in DEAD_COLUMNS if c in df.columns]
assert not leaked, f'dead columns present in the cleaned frame: {leaked}'
still_dead = [c for c in df.columns if df[c].nunique(dropna=True) <= 1 or all_blank(df[c])]
assert not still_dead, f'zero-variance or blank columns survived to the cleaned frame: {still_dead}'
print('all post-conditions passed')
print(f'dead columns absent from jobs_clean : {list(DEAD_COLUMNS)}')

df.to_parquet(OUT_DIR / 'jobs_clean.parquet', index=False)
job_category.to_parquet(OUT_DIR / 'job_category.parquet', index=False)
M['outputs'] = {'jobs_clean.parquet': len(df), 'job_category.parquet': len(job_category)}

print(f'\nwrote {OUT_DIR/"jobs_clean.parquet"}      {len(df):,} rows x {df.shape[1]} cols')
print(f'wrote {OUT_DIR/"job_category.parquet"}   {len(job_category):,} rows')
print(f'\nraw {M["raw"]["rows"]:,} rows / {M["raw"]["mem"]:,.1f} MB  ->  '
      f'clean {M["final"]["rows"]:,} rows / {M["final"]["mem"]:,.1f} MB '
      f'({(M["final"]["mem"]/M["raw"]["mem"]-1)*100:+.0f}%)')

                                             dtype  na_count  na_pct  n_unique    zeros
employmentTypes                           category         0    0.00         8        0
metadata_expiryDate                 datetime64[us]         0    0.00       449        0
metadata_isPostedOnBehalf                     bool         0    0.00         2        0
metadata_jobPostId                             str         0    0.00   1044587        0
metadata_newPostingDate             datetime64[us]         0    0.00       429        0
metadata_originalPostingDate        datetime64[us]         0    0.00       603        0
metadata_repostCount                          int8         0    0.00         3  1001862
metadata_totalNumberJobApplication           int32         0    0.00       369   656375
metadata_totalNumberOfView                   int32         0    0.00      1543   179121
minimumYearsExperience                        Int8         0    0.00        47   114451
numberOfVacancies               

all post-conditions passed
dead columns absent from jobs_clean : ['location', 'salary_currency', 'description', 'requirements', 'created_at']



wrote ../data/processed/jobs_clean.parquet      1,044,587 rows x 29 cols
wrote ../data/processed/job_category.parquet   1,767,785 rows

raw 1,048,585 rows / 401.7 MB  ->  clean 1,044,587 rows / 211.4 MB (-47%)


## 13 · Feature enrichment — align to the `src/pipeline/` schema

The enrichment logic lives in **`src/pipeline/feature_enrichment.py`**, not in this notebook — it is
production logic that the pipeline (or any other consumer) should be able to import rather than
reimplement, and it is far easier to test and review as a module. The notebook imports it and
applies it.

`feature_enrichment(df, job_category=None)` renames the raw MCF column names to the names the
production pipeline and `jobs` table use, and derives the feature columns
`src/pipeline/feature_engineer.py` adds — so the cleaned data can feed the existing dashboard
without a translation layer. `schema_report(enriched)` returns the missing/extra column lists used
by the assertions below.

> Editing the module while the kernel is live? Run `%load_ext autoreload` and `%autoreload 2`
> before the import cell, or restart the kernel — Python caches imported modules.

**Rename map** (raw source name → pipeline name), defined as `RENAME_MAP` in the module:

| source | pipeline | | source | pipeline |
|---|---|---|---|---|
| `metadata_jobPostId` | `job_id` | | `metadata_newPostingDate` | `posting_date` |
| `postedCompany_name` | `company` | | `metadata_expiryDate` | `expiry_date` |
| `primary_category` | `sector` | | `metadata_totalNumberOfView` | `views` |
| `salary_minimum` | `salary_min` | | `metadata_totalNumberJobApplication` | `applications` |
| `salary_maximum` | `salary_max` | | `numberOfVacancies` | `vacancies` |
| `average_salary` | `salary_midpoint` | | `metadata_repostCount` | `repost_count` |
| `positionLevels` | `position_level` | | `employmentTypes` | `job_type` |
| `minimumYearsExperience` | `seniority_years` | | | |

**Derived to match `feature_engineer.py`**: `experience_level`, `salary_band`, `skills`,
`skill_count`, `competitiveness_score`.

### Five `JOBS_SCHEMA` columns are deliberately **not** materialised

The pipeline creates these as constants because the `jobs` table declares them. This notebook does
not: writing the same value down a million rows is exactly the dead weight Step 4 removes from the
source side, and it would be inconsistent to prune `salary_type` for being constant and then invent
`salary_currency` two steps later. `DEAD_COLUMNS` in the setup cell names them with the reason each is dead — which columns are worth
materialising is a property of *this extract*, so the judgement is declared here with the other
cleaning decisions rather than inside the enrichment module. That module carries only the outcome:
the five are **absent from its `JOBS_SCHEMA_COLUMNS`** rather than skipped at build time, so
nothing can read the list and rebuild them. The full table definition still lives in
`src/database/schema.py`.

| column | pipeline would write | why it is dead |
|---|---|---|
| `location` | `'Singapore'` | the extract is Singapore-only — zero variance |
| `salary_currency` | `'SGD'` | restates `salary_type`, itself dropped as constant in Step 4 |
| `description` | `''` | the source CSV has no description field at all |
| `requirements` | `''` | the source CSV has no requirements field at all |
| `created_at` | `Timestamp.now()` | records when the notebook ran, not anything about the posting — and it makes the parquet's bytes change on every execution with no change in the data |

**Nothing downstream breaks.** `DatabaseManager.insert_jobs` already back-fills `created_at` and
`salary_currency` itself and filters `columns_order` to the columns actually present, naming
anything absent; `location`, `description` and `requirements` are nullable in `JOBS_SCHEMA`. The
facts do not disappear either — that all postings are Singapore, monthly SGD is stated in this
report, once, instead of a million times in the file.

The cell below asserts none of them reached the enriched frame, and Step 12 asserts the same of the
cleaned frame, so a future edit that reintroduces one fails the notebook rather than silently
growing the file.

Two deliberate deviations from the current pipeline, both flagged rather than silent:

1. **`job_id` carries `metadata_jobPostId`** instead of a fresh UUID, so enriched rows can still be
   joined back to the raw layer, to `job_category.parquet`, and to MCF. The pipeline's UUID makes
   that impossible.
2. **`competitiveness_score` uses the 99th percentile as its salary denominator**, not `max()`.
   The pipeline's `max()` denominator is dominated by the un-nulled outlier, which compresses the
   median row's salary contribution to ~0.02 of the 50 available points.

Two `feature_engineer.py` columns are **not** reproduced:

* **`days_posted`** — it is `now() - posting_date`, which measures when ingestion ran rather than
  anything about the posting, and it would make the saved parquet change on every execution.
  `listing_days` (`expiry_date - posting_date`) is retained instead: deterministic, and the
  duration anyone actually wants. Anything needing days-since-posting can compute it at query time
  against its own reference.
* **`is_growth_role`** — its definition (`count > median * 0.2`) marks essentially every role in
  the file.

Neither is in `JOBS_SCHEMA`; in fact of everything `feature_engineer.py` computes, only
`seniority_years` reaches the `jobs` table — `salary_midpoint`, `salary_band`, `skill_count`,
`competitiveness_score`, `days_posted` and `is_growth_role` are all dropped at load. They are kept
here anyway (except the two above) because the enriched parquet is meant to be usable directly,
not only as a DB feed.

The notebook's own provenance columns (`salary_flag`, `salary_na`, `dup_group_size`,
`is_same_day_dup`, `title_normalised`, `company_normalised`, `source`, `n_categories`,
`listing_days`) are retained — `jobs_enriched.parquet` is a superset of the `jobs` table, and a
loader can select just the `JOBS_SCHEMA` columns.

In [16]:
import sys

# Make the repo root importable regardless of where the kernel was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'pipeline').is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.pipeline.feature_enrichment import (
    JOBS_SCHEMA_COLUMNS,
    RENAME_MAP,
    feature_enrichment,
    schema_report,
)

print('imported from', (REPO_ROOT / 'src' / 'pipeline' / 'feature_enrichment.py').relative_to(REPO_ROOT))

enriched = feature_enrichment(df, job_category=job_category)

check = schema_report(enriched)
assert not check['missing'], f"schema columns still missing: {check['missing']}"
leaked = [c for c in DEAD_COLUMNS if c in enriched.columns]
assert not leaked, f'dead columns reached the enriched frame: {leaked}'
assert not [c for c in enriched.columns if all_blank(enriched[c])], 'a blank column reached the enriched frame'
assert enriched['job_id'].is_unique, 'job_id must stay unique to serve as the primary key'
assert len(enriched) == len(df), 'enrichment changed the row count'

M['enrichment'] = {
    'renamed': len(RENAME_MAP),
    'schema_cols': len(JOBS_SCHEMA_COLUMNS),
    'dead_cols': dict(DEAD_COLUMNS),
    'extra_cols': check['extra'],
    'sub_sector_filled': int(enriched['sub_sector'].notna().sum()),
    'skills_found': int((enriched['skills'] != 'Not Specified').sum()),
    'skill_count_mean': round(float(enriched['skill_count'].mean()), 3),
    'comp_median': round(float(enriched['competitiveness_score'].median()), 2),
    'module': 'src/pipeline/feature_enrichment.py',
}
print(f'renamed {len(RENAME_MAP)} columns; all {len(JOBS_SCHEMA_COLUMNS)} JOBS_SCHEMA columns present')
print(f'dead columns, excluded from the schema list and never built ({len(DEAD_COLUMNS)}): '
      f'{list(DEAD_COLUMNS)}')
print(f"extra columns retained beyond JOBS_SCHEMA ({len(check['extra'])}): {check['extra']}\n")

print('--- JOBS_SCHEMA-aligned preview ---')
print(enriched[['job_id', 'title', 'company', 'sector', 'sub_sector', 'salary_min', 'salary_max',
                'experience_level', 'seniority_years', 'job_type']].head(4).to_string(index=False))
print()
print('derived feature columns:')
print(enriched[['salary_midpoint', 'salary_band', 'skills', 'skill_count',
                'listing_days', 'competitiveness_score']].head(4).to_string(index=False))
print()
print(f'sub_sector populated  : {M["enrichment"]["sub_sector_filled"]:,} rows '
      f'({M["enrichment"]["sub_sector_filled"]/len(enriched)*100:.1f}%) - pipeline leaves this NULL')
print(f'skills matched        : {M["enrichment"]["skills_found"]:,} rows '
      f'({M["enrichment"]["skills_found"]/len(enriched)*100:.1f}%), mean skill_count '
      f'{M["enrichment"]["skill_count_mean"]}')
print(f'competitiveness median: {M["enrichment"]["comp_median"]} '
      f'(pipeline max()-denominator would give ~0.02 for the salary half)\n')

enriched.to_parquet(OUT_DIR / 'jobs_enriched.parquet', index=False)
M['outputs']['jobs_enriched.parquet'] = len(enriched)
print(f'wrote {OUT_DIR/"jobs_enriched.parquet"}   {len(enriched):,} rows x {enriched.shape[1]} cols')

compare(snap(df), enriched, 'Step 13 - feature enrichment',
        f'{len(RENAME_MAP)} columns renamed to the src/pipeline/ schema; '
        f'all {len(JOBS_SCHEMA_COLUMNS)} JOBS_SCHEMA columns satisfied, '
        f'{len(DEAD_COLUMNS)} dead ones excluded')

imported from src/pipeline/feature_enrichment.py


renamed 15 columns; all 18 JOBS_SCHEMA columns present
dead columns, excluded from the schema list and never built (5): ['location', 'salary_currency', 'description', 'requirements', 'created_at']
extra columns retained beyond JOBS_SCHEMA (17): ['metadata_isPostedOnBehalf', 'metadata_originalPostingDate', 'status_jobStatus', 'salary_midpoint', 'n_categories', 'salary_na', 'salary_flag', 'title_normalised', 'company_normalised', 'listing_days', 'is_repost', 'source', 'dup_group_size', 'is_same_day_dup', 'salary_band', 'skill_count', 'competitiveness_score']

--- JOBS_SCHEMA-aligned preview ---
          job_id                                                          title                    company                 sector    sub_sector  salary_min  salary_max experience_level  seniority_years  job_type
MCF-2023-0252866      Food Technologist - Clementi | Entry Level | Up to $2,800        WORKSTONE PTE. LTD.   Environment / Health Manufacturing        2000        2800      Entry Level    

wrote ../data/processed/jobs_enriched.parquet   1,044,587 rows x 35 cols
=== Step 13 - feature enrichment ===
    15 columns renamed to the src/pipeline/ schema; all 18 JOBS_SCHEMA columns satisfied, 5 dead ones excluded
               rows   cols    mem MB    NaN cells
BEFORE    1,044,587     29     211.4       28,731
AFTER     1,044,587     35     245.6      683,682
DELTA            +0     +6     +34.2     +654,951
    columns removed: ['employmentTypes', 'metadata_expiryDate', 'metadata_jobPostId', 'metadata_newPostingDate', 'metadata_repostCount', 'metadata_totalNumberJobApplication', 'metadata_totalNumberOfView', 'minimumYearsExperience', 'numberOfVacancies', 'positionLevels', 'postedCompany_name', 'salary_maximum', 'salary_minimum', 'average_salary', 'primary_category']
    columns added  : ['job_type', 'expiry_date', 'job_id', 'posting_date', 'repost_count', 'applications', 'views', 'seniority_years', 'vacancies', 'position_level', 'company', 'salary_max', 'salary_min', 'salary_

{'rows': 1044587,
 'cols': 35,
 'mem': np.float64(245.6),
 'na': 683682,
 'columns': ['job_type',
  'expiry_date',
  'metadata_isPostedOnBehalf',
  'job_id',
  'posting_date',
  'metadata_originalPostingDate',
  'repost_count',
  'applications',
  'views',
  'seniority_years',
  'vacancies',
  'position_level',
  'company',
  'salary_max',
  'salary_min',
  'status_jobStatus',
  'title',
  'salary_midpoint',
  'sector',
  'n_categories',
  'salary_na',
  'salary_flag',
  'title_normalised',
  'company_normalised',
  'listing_days',
  'is_repost',
  'source',
  'dup_group_size',
  'is_same_day_dup',
  'sub_sector',
  'experience_level',
  'salary_band',
  'skills',
  'skill_count',
  'competitiveness_score']}

## 14 · Generate the markdown report

Assembles `docs/data-cleaning-report-generated.md` from the `M` metrics collected above. Prose
carries the justification; every number is interpolated from the run, so the report cannot drift
away from the data.

In [17]:
p = M['params']
# The value each dead column would have held, for the section 6 table.
DEAD_COL_VALUES = {'location': "`'Singapore'` on every row", 'salary_currency': "`'SGD'` on every row",
                   'description': '`\'\'` (empty string)', 'requirements': '`\'\'` (empty string)',
                   'created_at': '`Timestamp.now()` at run time'}
steps_tbl = to_md(pd.DataFrame([
    {'step': s['step'], 'rows': f'{s["rows_before"]:,} -> {s["rows_after"]:,}',
     'cols': f'{s["cols_before"]} -> {s["cols_after"]}',
     'mem MB': f'{s["mem_before"]:,.0f} -> {s["mem_after"]:,.0f}'} for s in M['steps']]), index=False)

R = []
A = R.append

A('# Data Cleaning Report - SGJobData')
A('')
A(f'Generated by `notebooks/data_cleaning.ipynb`. Source: `{RAW_CSV}`. '
  'Every figure is computed at run time from the data itself.')
A('')
A(f'| | rows | cols | memory (MB) | NaN cells |')
A(f'| --- | ---: | ---: | ---: | ---: |')
A(f'| raw | {M["raw"]["rows"]:,} | {M["raw"]["cols"]} | {M["raw"]["mem"]:,.1f} | {M["raw"]["na"]:,} |')
A(f'| clean | {M["final"]["rows"]:,} | {M["final"]["cols"]} | {M["final"]["mem"]:,.1f} | {M["final"]["na"]:,} |')
A('')
A(f'**{M["raw"]["rows"] - M["final"]["rows"]:,} rows removed '
  f'({(1 - M["final"]["rows"]/M["raw"]["rows"])*100:.2f}%). '
  f'Memory {(M["final"]["mem"]/M["raw"]["mem"] - 1)*100:+.0f}%. '
  f'Cells imputed: {M["filled_cells"]}.**')
A('')
A('### Step ledger')
A('')
A(steps_tbl)
A('')
A('### Parameters')
A('')
A(f'| parameter | value | meaning |')
A(f'| --- | --- | --- |')
A(f'| `SALARY_FLOOR` | {p["SALARY_FLOOR"]:,} | monthly SGD at or below which a salary is a placeholder - **nulled** |')
A(f'| `SALARY_CEILING` | {p["SALARY_CEILING"]:,} | monthly SGD above which a salary is a statistical outlier - **flagged, not nulled** |')
A(f'| `INTERN_STIPEND_FLOOR` | {p["INTERN_STIPEND_FLOOR"]:,} | monthly SGD below which even an internship stipend is implausible |')
A(f'| `EXPERIENCE_MAX` | {p["EXPERIENCE_MAX"]} | years above which the value is impossible |')
A(f'| `SYNTHETIC_ID_RE` | `{p["SYNTHETIC_ID_RE"]}` | job-ID pattern marking generated test rows |')
A('')
A('---')
A('')

# ---- 1. type conversions -----------------------------------------------------
A('## 1. Column type conversions - and where data would be lost')
A('')
A(f'### 1.1 Dates: `str` -> `{M["date_dtype"]}`  *(lossless, asserted)*')
A('')
A(M['md_dates'])
A('')
A('**Justification.** As strings these columns cannot do date arithmetic, resampling, or `.dt` '
  'access - every time-series view in the dashboard needs real datetimes. **No data loss:** the '
  'notebook refuses the conversion unless `strftime` round-trips to the original string on every '
  'row, which it does. No timezone is applied, because the source has no time component and '
  'localising would imply precision that does not exist.')
A('')
A(f'Cross-field logic also holds: `originalPostingDate > newPostingDate` in '
  f'**{M["date_checks"]["orig_after_new"]}** rows, `expiryDate <= newPostingDate` in '
  f'**{M["date_checks"]["expiry_before_post"]}** rows. Median lifespan '
  f'{M["date_checks"]["lifespan_median"]:.0f} days.')
A('')
A('### 1.2 `categories` JSON -> bridge table  *(lossless as done; lossy if shortcut)*')
A('')
A(f'- {M["categories"]["distinct"]} distinct categories, '
  f'{M["categories"]["assignments"]:,} assignments over {M["final"]["rows"]:,} postings')
A('- categories per posting: ' + ', '.join(
    f'{k} -> {v:,} postings' for k, v in M['categories']['per_row'].items()))
A(f'- {M["categories"]["multi_pct"]}% of postings carry more than one category')
A('')
A(M['md_top_categories'])
A('')
A(f'**Justification.** A JSON string cannot be filtered on - "all IT jobs" would need a substring '
  f'match that also catches the label inside unrelated text. The many-to-many relationship is '
  f'preserved in `job_category.parquet` ({M["categories"]["assignments"]:,} rows). '
  f'**This is where loss would occur if shortcut:** keeping only the first category would discard '
  f'{M["categories"]["lost_if_first_only"]:,} assignments and systematically understate every '
  f'secondary category. `primary_category` is kept on the wide table as a convenience only.')
A('')
A('### 1.3 Categorical and integer conversions  *(lossless, range-asserted)*')
A('')
A(M['md_types'])
A('')
A('**Justification.** Eight distinct employment types over a million rows is the textbook case for '
  '`category`: the string is stored once and each row holds a small integer code. Every integer '
  'downcast is asserted against the observed range before being applied, so the loop fails loudly '
  'rather than wrapping silently. The two `metadata_total*` counters are given `int32` rather than '
  'the tighter `int16` their current range permits - `int16` leaves only ~24k headroom and one '
  'viral posting in a future extract would overflow it.')
A('')
A('### 1.4 `average_salary` -> `float32`: the conversion whose answer depends on when you ask')
A('')
A(f'| measured at | max value | max round-trip error |')
A(f'| --- | ---: | ---: |')
A(f'| raw file | ${M["float32_raw"]["max_value"]:,.0f} | **{M["float32_raw"]["max_error"]}** |')
A(f'| after synthetic rows removed | ${M["float32_before"]["max_value"]:,.0f} | '
  f'{M["float32_before"]["max_error"]} |')
A(f'| after the Step 7 salary fix | ${M["float32"]["max_value"]:,.0f} | '
  f'{M["float32"]["max_error"]} |')
A('')
if M['float32_raw']['max_error'] > 0:
    A(f'**On the raw file the cast is lossy** - error {M["float32_raw"]["max_error"]}, because '
      f'float32 carries only ~7 significant digits and the multi-million synthetic salaries land '
      f'beyond the range where it can represent a `.5`. '
      f'{M["float32"]["half_values"]:,} rows carry a genuine `.5` fraction (odd min+max), so this '
      f'is a real error, not a display artefact. Remove the synthetic rows and the same cast '
      f'becomes exact.')
    A('')
    A('The point is not that float32 is unusable - it is that "is this conversion lossless?" has '
      'no answer independent of the cleaning order. Profiling dtypes on a dirty frame gives a '
      'different verdict than profiling the same frame after junk-row removal, which is why every '
      'dtype decision in this notebook is made last.')
else:
    A(f'The cast is exact at every stage on this extract, including the raw file '
      f'({M["float32"]["half_values"]:,} rows carry a genuine `.5` fraction, all within float32 '
      f'precision).')
A('')
A(f'It is **kept as `Float64` regardless**: the saving is only a few MB, and float32 would '
  f'silently re-break the moment a future extract reintroduces large values. `average_salary` is '
  f'also fully derived from `salary_minimum`/`salary_maximum`, so it is recomputed after the '
  f'salary fixes rather than trusted - a stored derived column drifts out of sync with its inputs.')
A('')
A('### 1.5 Columns dropped as dead')
A('')
A(f'| column | reason |')
A(f'| --- | --- |')
for c in M['dead_cols']['empty']:
    A(f'| `{c}` | 100% NaN, 0 distinct values - empty column |')
for c, v in M['dead_cols']['constant'].items():
    A(f'| `{c}` | single value `{v}` on every row - zero variance |')
A('')
A('**Justification.** A constant column cannot correlate with anything and cannot be filtered on '
  'meaningfully. `salary_type` being constant is itself information - *all salaries are monthly '
  'SGD* - and belongs in this report rather than repeated a million times in the frame. '
  '`metadata_isPostedOnBehalf` is retained despite its 94/6 split: low variance is not zero '
  'variance, and it flags recruiter-posted listings, which matters for the duplicate analysis in '
  'section 5.')
A('')
A('---')
A('')

# ---- 2. ghost rows -----------------------------------------------------------
A('## 2. Ghost rows removed')
A('')
A(f'**{M["ghost"]["n"]:,} rows ({M["ghost"]["pct"]}% of the file) - all removed.**')
A('')
A(f'| evidence | value |')
A(f'| --- | --- |')
A(f'| rows where all {M["ghost"]["text_cols"]} text columns are NaN | {M["ghost"]["n"]:,} |')
A(f'| rows with a *partial* NaN pattern | **{M["ghost"]["partial"]}** |')
A(f'| observed NaN-count-per-row values | {M["ghost"]["bimodal"]} |')
A(f'| sum of abs() over every numeric column on these rows | **{M["ghost"]["numeric_abs_sum"]}** |')
A(f'| rows with 0 vacancies outside this group | {M["ghost"]["vacancies_zero_elsewhere"]} |')
A(f'| index span | {M["ghost"]["idx_min"]:,} - {M["ghost"]["idx_max"]:,} |')
A('')
A('**Justification for deletion rather than imputation.** The per-row NaN count is strictly '
  'bimodal - a row is either fully populated or fully blank, never in between - and every numeric '
  'field on the blank rows is exactly zero. There is nothing to impute *from*: no title, no '
  'company, no ID, no dates. Every real posting has at least one vacancy; a record with zero '
  'vacancies, zero salary and no identifier is not a job. Imputing would fabricate '
  f'{M["ghost"]["n"]:,} synthetic postings and inflate every count on the dashboard. At '
  f'{M["ghost"]["pct"]}% of the data, dropping costs nothing statistically.')
A('')
A('**Ordering trap.** `occupationId` is 100% NaN, so on the raw frame `df.dropna()` returns an '
  '**empty** DataFrame. The notebook uses an explicit all-text-NaN mask instead, which states the '
  'intent and cannot misfire.')
A('')
A(f'### Synthetic test rows: {M["synthetic"]["n"]} more removed')
A('')
A(f'Rows matching `{p["SYNTHETIC_ID_RE"]}` are generated test data - the IDs embed a generation '
  f'timestamp and salaries reach ${M["synthetic"]["max_salary"]:,}/month. Removed by ID pattern, '
  f'not by index, so the filter survives a reload. ID prefixes present in the raw file: '
  f'{M["synthetic"]["prefixes"]}. The `ATS-` rows are legitimate (real companies, sane salaries) '
  f'and are retained, tagged via the new `source` column: {M["source_mix"]}.')
A('')
A('---')
A('')

# ---- 3. data fixing ----------------------------------------------------------
A('## 3. Data fixing, per column')
A('')
A('### 3.1 `salary_minimum` / `salary_maximum` / `average_salary`')
A('')
A('**Before**')
A('')
A(M['md_salary_before'])
A('')
A('**After**')
A('')
A(M['md_salary_after'])
A('')
A(f'**The floor and ceiling are handled differently on purpose** - not two symmetric guard rails, '
  f'but two different kinds of problem:')
A('')
A(f'| defect | rows | treatment |')
A(f'| --- | ---: | --- |')
A(f'| `salary_minimum < {p["SALARY_FLOOR"]:,}` (undisclosed placeholder) | '
  f'{M["salary_fix"]["low_min_n"]:,} | **null** the pair |')
A(f'| `salary_maximum < {p["SALARY_FLOOR"]:,}` (undisclosed placeholder) | '
  f'{M["salary_fix"]["low_max_n"]:,} | **null** the pair |')
A(f'| of which exactly `$1 - $1` | {M["salary_fix"]["sentinel_exactly_1"]:,} | **null** the pair |')
A(f'| low minimum but plausible maximum (e.g. `$1 - $600`) | '
  f'{M["salary_fix"]["min_only_n"]:,} | **null** the pair |')
A(f'| internship, both bounds in `[{p["INTERN_STIPEND_FLOOR"]:,}, {p["SALARY_FLOOR"]:,})` | '
  f'{M["salary_fix"]["intern_stipend_n"]:,} | **keep**, flag `low_stipend` |')
A(f'| `salary_maximum > {p["SALARY_CEILING"]:,}` (statistical outlier) | '
  f'{M["salary_fix"]["high_n"]:,} | **keep, do not null** - flag `outlier` |')
A(f'| `salary_minimum > salary_maximum` | {M["salary_fix"]["inverted"]} | none needed |')
A('')
A(f'**Why the asymmetry.** `$1/month` and a `$10`-`$15` hourly rate typed into a monthly-only '
  f'field are not real numbers at *any* resolution - there is nothing to preserve, so the floor '
  f'defect is nulled. A `${p["SALARY_CEILING"]:,}+`/month posting is different: it might be a '
  f'genuine C-suite salary or it might be a missing decimal, and which one is true depends on the '
  f'question being asked of the data - a mean/std view wants it excluded, a "highest-paid roles" '
  f'or fraud-detection view wants to see it. That is an analysis-stage judgment, not a '
  f'cleaning-stage fact, so the raw value is left untouched and only a `salary_flag == \'outlier\'` '
  f'marker is added. Whether to filter on it is left to whoever runs the query.')
A('')
A(f'**The pair is the unit of validity for the floor check.** A quoted salary is a *range*, so if '
  f'either bound is a placeholder the whole range is untrustworthy. Nulling only `salary_maximum` '
  f'would leave {M["salary_fix"]["min_only_n"]:,} rows like `$1 - $600` behind, whose average of '
  f'$300.50 sits below the very floor the rule is meant to enforce. Both bounds are therefore '
  f'nulled together, and a post-condition asserts they are never nulled apart.')
A('')
A(f'**Justification for the floor.** $1/month is not a wage - it is what a poster enters to satisfy '
  f'a required field. Singapore has no statutory minimum wage, so there is no bright line; '
  f'${p["SALARY_FLOOR"]:,} is a conservative "cannot be a real monthly wage" threshold well below '
  f'the Progressive Wage Model floors (~$1,400-1,600). Note the low group skews part-time and '
  f'contract ({M["salary_fix"]["low_by_employment"]}), so some are hourly rates forced into a '
  f'monthly-only field - mis-unit rather than undisclosed, but both must be excluded from salary '
  f'aggregates.')
A('')
A(f'**Internship carve-out.** A flat ${p["SALARY_FLOOR"]:,} floor risked deleting genuine low '
  f'stipends, so it was checked before being applied rather than assumed. Internship salaries in '
  f'this file have a median of $1,200 and a 5th percentile of $800 - almost none are legitimately '
  f'this low - but {M["salary_fix"]["intern_stipend_n"]:,} rows tagged `Internship/Attachment` '
  f'with both bounds in `[{p["INTERN_STIPEND_FLOOR"]:,}, {p["SALARY_FLOOR"]:,})` are real stipends, '
  f'not placeholders. They are kept and marked `low_stipend` instead of `undisclosed`, visible to '
  f'any analysis that wants to exclude them, invisible to one that does not - the same '
  f'flag-don\'t-null principle used for the ceiling, applied at the low end.')
A('')
A(f'**Why not the textbook IQR rule** for a ceiling, if one were applied at cleaning time at all. '
  f'Q1=${M["salary_iqr"]["q1"]:,.0f}, Q3=${M["salary_iqr"]["q3"]:,.0f}, so even the lenient 3xIQR '
  f'upper fence sits at ${M["salary_iqr"]["fence_3iqr"]:,.0f} and would catch '
  f'{M["salary_iqr"]["rows_above_fence"]:,} rows - tens of thousands of legitimate senior roles. '
  f'Whatever threshold an analysis chooses to filter `salary_flag == \'outlier\'` on, it should not '
  f'be the IQR rule.')
A('')
A(f'Total nulled: **{M["salary_fix"]["nulled"]:,}** rows (floor only). Remaining salary coverage: '
  f'**{M["salary_fix"]["coverage_pct"]}%** - higher than if the ceiling were also nulled, because '
  f'those {M["salary_fix"]["high_n"]:,} rows are still present, just flagged. `average_salary` is '
  f'recomputed from the cleaned inputs rather than inherited, which means it **still contains the '
  f'flagged outliers** (max ${M["salary_fix"]["max_before"]:,}) until an analysis filters them out - '
  f'that is the point, not an oversight. Two flags are built rather than a bare NaN: `salary_na` '
  f'for a quick boolean filter on the nulled rows, and `salary_flag` '
  f'(`{M["salary_fix"]["flag_counts"]}`) recording *why* - `undisclosed`, `outlier`, or the retained '
  f'`low_stipend` carve-out - so the reason is never lost along with the value.')
A('')
A('### 3.2 `minimumYearsExperience`')
A('')
A(f'Observed max was **{M["experience"]["max_before"]}** years; tail values '
  f'{M["experience"]["tail"]}. Values above {p["EXPERIENCE_MAX"]} are physically impossible and '
  f'were set to NaN - **{M["experience"]["impossible"]} rows**, too few to move any aggregate. '
  f'These are poster-side typos, not a pipeline defect, which is why the column is capped rather '
  f'than rebuilt. `0` is left untouched (see section 4).')
A('')
A('### 3.3 `title` and `postedCompany_name`')
A('')
A(f'| fix | before | after |')
A(f'| --- | ---: | ---: |')
A(f'| rows with stray whitespace in `title` | {M["text"]["title_ws_rows"]:,} | 0 |')
A(f'| distinct titles | {M["text"]["title_distinct_raw"]:,} | '
  f'{M["text"]["title_distinct_normalised"]:,} (normalised) |')
A(f'| distinct companies | {M["text"]["company_distinct_raw"]:,} | '
  f'{M["text"]["company_distinct_normalised"]:,} (normalised) |')
A('')
A(f'**Justification.** Whitespace is stripped and internal runs collapsed on `title` in place; '
  f'case-folding is kept in a separate `title_normalised` column so the display string survives. '
  f'Case-folding alone collapses **{M["text"]["title_collapsed"]:,}** distinct titles - without '
  f'it, "Software Engineer" and "software engineer" are counted as different roles in every '
  f'top-titles chart. `company_normalised` (upper + whitespace-collapsed) is added as a join key. '
  f'Legal-suffix variation (`PTE. LTD.` vs `PTE LTD`) is left alone deliberately: that is entity '
  f'resolution, not cleaning, and needs its own reviewed pass.')
A('')
A('### 3.4 Columns needing no fix')
A('')
A(f'`categories` parses cleanly on every row with {M["categories"]["empty_arrays"]} empty arrays. '
  f'`numberOfVacancies` spans 1-999 with no zeros; 999 looks like a UI cap rather than a true '
  f'count, which is worth a footnote on any "total vacancies" headline but is not a defect to '
  f'repair. `status_jobStatus`, `employmentTypes` and `positionLevels` have small clean value sets.')
A('')
A('---')
A('')

# ---- 4. data filling ---------------------------------------------------------
A('## 4. Data filling, per column')
A('')
A(f'**{M["filled_cells"]} cells were imputed.** After ghost-row removal the frame has no explicit '
  f'NaN outside the salaries this pipeline deliberately created. The filling question is really '
  f'about *implicit* missingness encoded as `0`:')
A('')
A(M['md_zeros'])
A('')
A('### Why nothing is filled')
A('')
A(f'**`metadata_totalNumberJobApplication`** - zero is genuinely bimodal: it means "nobody applied" '
  f'for many listings and "the counter was not populated at scrape time" for others, and the data '
  f'offers no way to separate them. Mean-filling would fabricate applications; NaN-ing all of them '
  f'would discard the majority of the column. Left as `0`, flagged here, and application-rate '
  f'metrics should be restricted to rows with views > 0.')
A('')
A(f'**`metadata_totalNumberOfView`** - same ambiguity at a lower rate. A posting with 0 views and 0 '
  f'applications is internally consistent, so view-based analysis on the non-zero subset is '
  f'defensible. Guard the division: `applications / views` is undefined for these rows.')
A('')
A(f'**`minimumYearsExperience`** - `0` is a real value meaning "no prior experience required". '
  f'This was tested rather than assumed - the seniority mix of the zero-experience rows against '
  f'the frame overall:')
A('')
A(M['md_zero_exp_mix'])
A('')
A('The zeros concentrate at the junior end rather than spreading evenly across seniority, which is '
  'what a defaulted-to-zero field would look like. They are real.')
A('')
A(f'**Salaries** - the ~{M["salary_fix"]["nulled"]:,} nulled values (the floor defect only) are '
  f'**not** imputed by median, group median, or regression. Salary is the dependent variable this '
  f'dashboard exists to measure; filling it with a group median manufactures the very distribution '
  f'being observed and tightens variance so every confidence interval comes out wrong. Exclude from '
  f'aggregates and report the {M["salary_fix"]["coverage_pct"]}% coverage rate alongside every '
  f'salary figure. If a downstream model cannot accept NaN, add an explicit `salary_imputed` flag '
  f'so the imputation is never invisible.')
A('')
A(f'The {M["salary_fix"]["high_n"]:,} ceiling outliers are a related but separate case: they are '
  f'not missing, so there is nothing to fill. `salary_flag == \'outlier\'` marks them and leaves '
  f'the decision to exclude, cap, or report them separately to whichever analysis consumes the '
  f'data - see section 3.1 for why that choice is deferred rather than made here.')
A('')
A('### Derived columns added instead of filling')
A('')
A('| column | definition |')
A('| --- | --- |')
for k, v in M['derived'].items():
    A(f'| `{k}` | {v} |')
A('')
A('---')
A('')

# ---- 5. duplicates -----------------------------------------------------------
A('## 5. Duplicate rows')
A('')
A(M['md_duplicates'])
A('')
A(f'**No rows were dropped as duplicates.** They are flagged via `dup_group_size` and '
  f'`is_same_day_dup` so the decision belongs to the analysis that consumes the data.')
A('')
A('### Justification')
A('')
A(f'**Keys 1 and 2 are clean.** {M["dup"]["exact"]} exact duplicate rows and {M["dup"]["pk"]} '
  f'duplicate `metadata_jobPostId` - the primary key is sound and the loader is not '
  f'double-reading anything. Any deduplication beyond this point is a judgement about business '
  f'meaning, not a repair.')
A('')
A(f'**Key 3 ({M["dup"]["content"]:,} rows) is serial reposting, not duplication.** The same role '
  f'posted by the same agency on different dates with different job IDs is how recruitment '
  f'agencies work. Each is a real posting that really appeared on the platform. Collapsing them '
  f'would erase the time dimension of hiring demand - exactly what a job-market dashboard is '
  f'measuring.')
A('')
A(f'**Key 4 ({M["dup"]["same_day"]:,} rows) is the only arguable case - and the evidence says '
  f'keep.** These are {M["dup"]["same_day_rows"]:,} rows in {M["dup"]["same_day_groups"]:,} groups '
  f'of identical same-day postings (largest group: {M["dup"]["largest_group"]:,}). The decisive '
  f'test: **{M["dup"]["groups_views_differ"]:,} of {M["dup"]["same_day_groups"]:,} groups have '
  f'different view counts** across their members, and {M["dup"]["groups_apps_differ"]:,} have '
  f'different application counts. Distinct traffic means the platform served them as separate '
  f'listings that jobseekers found and viewed separately. Dropping them would discard real '
  f'engagement data.')
A('')
A(f'They are also concentrated: {M["dup"]["behalf_in_dups"]}% are `isPostedOnBehalf` against a '
  f'{M["dup"]["behalf_baseline"]}% baseline, and the top contributors are manpower agencies.')
A('')
A(M['md_dup_agencies'])
A('')
A('**Recommended handling downstream.** Keep every row for volume and time-series work. When '
  'ranking employers or estimating distinct job openings, deduplicate on key 4 at query time - '
  'reversible, and the choice stays visible in the analysis rather than being baked into the '
  'stored dataset. `numberOfVacancies` is tracked separately, so a company posting five identical '
  'roles is not the same as one posting with five vacancies.')
A('')
A('---')
A('')

# ---- 6. enrichment -----------------------------------------------------------
e = M['enrichment']
A('## 6. Feature enrichment - alignment with `src/pipeline/`')
A('')
A(f'Implemented in **`{e["module"]}`** and imported by the notebook, so the pipeline and any other '
  f'consumer can use the same logic rather than reimplementing it.')
A('')
A(f'`feature_enrichment(df, job_category=None)` renames **{e["renamed"]}** source columns to the '
  f'names used by `src/pipeline/` and derives the feature columns `feature_engineer.py` adds, so '
  f'all **{e["schema_cols"]}** `JOBS_SCHEMA` columns it produces are satisfied and the cleaned '
  f'data can feed the existing dashboard without a translation layer. A further '
  f'**{len(e["dead_cols"])}** schema columns are dead: they are excluded from the module\'s '
  f'column list *and* never built (below). `schema_report()` from the same module backs the '
  f'assertions that fail the notebook if a live column is missing or a dead one reappears.')
A('')
A('| derived column | source | matches pipeline? |')
A('| --- | --- | --- |')
A('| `experience_level` | banded from `seniority_years` | yes - same thresholds |')
A('| `salary_band` | banded from `salary_max` | yes - same thresholds |')
A('| `skills`, `skill_count` | regex over `title` | yes - same vocabulary and boundary rule |')
A('| `sub_sector` | 2nd category from the bridge table | **no** - pipeline leaves it NULL |')
A('| `competitiveness_score` | p99 salary denominator | **no** - pipeline uses `max()` |')
A('| `job_id` | `metadata_jobPostId` | **no** - pipeline mints a fresh UUID |')
A('')
A(f'### {len(e["dead_cols"])} `JOBS_SCHEMA` columns deliberately not materialised')
A('')
A('These are absent from `JOBS_SCHEMA_COLUMNS` in `feature_enrichment.py`, so no caller can read '
  'the list and rebuild them; the full table definition still lives in `src/database/schema.py`.')
A('')
A('| column | value it would hold | why it is dead |')
A('| --- | --- | --- |')
for c, why in e['dead_cols'].items():
    A(f'| `{c}` | {DEAD_COL_VALUES[c]} | {why} |')
A('')
A('**Justification.** This is section 1.5\'s rule applied to the output side. Pruning '
  '`salary_type` for being constant and then inventing `salary_currency` two steps later would be '
  'incoherent: both encode the single fact *all salaries are monthly SGD*, which belongs in this '
  'report rather than in a million rows. `description` and `requirements` are worse than constant '
  '- the source CSV has no such fields, so they would be empty strings that read as "no '
  'requirements listed" rather than "never collected". `created_at` records when this notebook '
  'ran, not anything about the posting, and it would change the parquet\'s bytes on every '
  'execution with no change in the data.')
A('')
A('**Nothing downstream breaks.** `DatabaseManager.insert_jobs` back-fills `created_at` and '
  '`salary_currency` itself and filters `columns_order` to the columns actually present, printing '
  'anything absent; `location`, `description` and `requirements` are nullable in `JOBS_SCHEMA`. '
  'Both outputs are asserted free of these columns, and of any all-blank column, before they are '
  'written.')
A('')
A(f'**Two deliberate deviations**, both documented rather than silent:')
A('')
A(f'1. **`job_id` carries the real `metadata_jobPostId`.** The pipeline regenerates it as a UUID '
  f'at load time, which makes the processed rows impossible to join back to the raw layer or to '
  f'MCF - the audit trail the raw layer exists to provide becomes unreachable. Keeping the source '
  f'ID also lets `jobs_enriched.parquet` join to `job_category.parquet`.')
A(f'2. **`competitiveness_score` divides by the 99th percentile, not `max()`.** With the outlier '
  f'left in place (by design - see section 3.1), a `max()` denominator is dominated by it and '
  f'compresses the median row\'s salary contribution to roughly 0.02 of the 50 points available. '
  f'The p99 denominator gives a median score of **{e["comp_median"]}**.')
A('')
A('### Two `feature_engineer.py` columns deliberately not reproduced')
A('')
A('**`days_posted`** is `now() - posting_date`. That measures when the ingestion process happened, '
  'not any property of the job posting, and it would make the saved parquet change on every '
  'execution - the file\'s bytes shifting with no change in the data. On a 2023-24 extract it '
  'resolves to the same value for every row sharing a posting date, so it carries no information '
  'beyond `posting_date`, which is already a column. `listing_days` '
  '(`expiry_date - posting_date`) is retained instead: deterministic, and the duration an analysis '
  'would actually ask for. Anything needing days-since-posting can compute it at query time '
  'against its own reference date.')
A('')
A('**`is_growth_role`** uses `count > median * 0.2`, which marks essentially every role in the '
  'file.')
A('')
A('Neither is part of `JOBS_SCHEMA`. Worth noting more broadly: of everything '
  '`feature_engineer.py` computes, **only `seniority_years` actually reaches the `jobs` table** - '
  '`salary_midpoint`, `salary_band`, `skill_count`, `competitiveness_score`, `days_posted` and '
  '`is_growth_role` are all dropped by `columns_order` at load time. The remaining ones are kept '
  'in `jobs_enriched.parquet` regardless, because it is meant to be usable directly for analysis '
  'rather than only as a database feed.')
A('')
A(f'Coverage of the derived columns: `sub_sector` populated on '
  f'**{e["sub_sector_filled"]:,}** rows ({e["sub_sector_filled"]/M["final"]["rows"]*100:.1f}%), '
  f'`skills` matched on **{e["skills_found"]:,}** rows '
  f'({e["skills_found"]/M["final"]["rows"]*100:.1f}%, mean `skill_count` '
  f'{e["skill_count_mean"]}). The skills rate is low because **the source CSV has no description '
  f'column** - which is also why `description` is a dead column above - so the skill regex only '
  f'ever sees job titles. That limitation is inherited from the source data, but it means '
  f'`skill_count` and anything derived from it should be read as a title-keyword signal rather '
  f'than a requirements analysis.')
A('')
A(f'`jobs_enriched.parquet` carries {len(e["extra_cols"])} provenance columns beyond the `jobs` '
  f'table (`{"`, `".join(e["extra_cols"][:6])}`, ...) so a loader can `SELECT` just the '
  f'`JOBS_SCHEMA` columns while the cleaning decisions stay inspectable - and none of the '
  f'{len(e["dead_cols"])} dead ones.')
A('')
A('---')
A('')

# ---- appendix ----------------------------------------------------------------
A('## Appendix - final column profile')
A('')
A(M['md_profile_final'])
A('')
A('### Outputs')
A('')
for f, n in M['outputs'].items():
    A(f'- `data/processed/{f}` - {n:,} rows')

REPORT_PATH.write_text('\n'.join(R))
print(f'wrote {REPORT_PATH}  ({len(R)} lines, {REPORT_PATH.stat().st_size/1024:.1f} KB)')

wrote ../docs/data-cleaning-report-generated.md  (276 lines, 29.6 KB)


---

**Next step:** the analysis layer reads `data/processed/jobs_enriched.parquet` (pipeline column
names, joinable to `job_category.parquet` on `job_id`) or `jobs_clean.parquet` for the source
names. Salary aggregates must filter `salary_na == False` **and** `salary_flag != 'outlier'` and
quote the coverage rate; employer rankings should deduplicate on the same-day key at query time.